**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**03 | Participant-Grouped Classical and Deep Unimodal Baselines**

![Participant-Grouped Classical and Deep Unimodal Baselines workflow](../../images/03_unimodal_grouped_baselines.svg)

**Research objective:** Compare source-aligned sEMG-only and acceleration-only classifiers on identical previously unseen Development participants.

**Dataset:** Paired E1 windows and person-independent assignments from Notebook 02.

**Notebook scope:** Run a training-prior comparator, Logistic Regression, Random Forest, 1D-CNN, TCN and BiGRU without fitting or selecting on Test people.

**00. Colab Repository Setup**

In [1]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


/content/human-movement-intelligence


**01. Imports and Research Environment**

In [2]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ---------------------------- 2. Standard Library ----------------------------
import time

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# -------------------- 5. Deep Learning and Compute Device --------------------
import torch
from core.computer_device import computer_device
device = computer_device(min_vram_gb=None)

# ------------------------ 6. Study Research Utilities ------------------------
from ninapro_dataset import ensure_numerical_cache
from ninapro_evaluation import (
    build_classical, classical_features, aligned_probabilities,
    participant_metrics, select_model, predictions_from_probabilities,
)
from ninapro_models import UNIMODAL_MODELS
from ninapro_training import fit_neural, predict_neural

# ---------------------- 7. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


Local changes kept; continuing with the current checkout. Conflicting Git update deferred.
Compute device and system information
OS: Linux 6.6.159+
CPU: x86_64
Installed RAM: 12.7 GiB
NVIDIA GPU: Tesla T4
Total VRAM: 15.00 GiB
PyTorch version: 2.14.1+cu130
CUDA device: Tesla T4
CUDA runtime: 13.0
cuDNN version: 92800
PyTorch total VRAM: 14.56 GiB
Selected compute device: cuda
Research output directory: /content/human-movement-intelligence/multimodal_movement_learning/output


**02. Read Fixed Classes and Paired Development Windows**

Notebook 02 supplies unique window rows, source labels, preserved person roles and fixed Development folds. The reserved Test people are excluded from every model-development operation. In a fresh Colab kernel, the numerical source cache can be reconstructed from original participant archives without changing the canonical window index.

In [3]:
# =========== 02. Read Fixed Classes and Paired Development Windows ===========

NOTEBOOK_NUMBER = 3
index = pd.read_csv(output_dir / "02_window_index.csv")
roles = pd.read_csv(output_dir / "02_participant_split_assignments.csv")
folds = pd.read_csv(output_dir / "02_development_fold_assignments.csv")
classes = np.sort(pd.read_csv(output_dir / "01_label_dictionary.csv").label.to_numpy(int))
role_map = dict(zip(roles.subject_id, roles.role))
fold_map = dict(zip(folds.subject_id, folds.validation_fold))
index["role"] = index.subject_id.map(role_map)
index["validation_fold"] = index.subject_id.map(fold_map)
if index.role.isna().any() or index.loc[index.role == "Development", "validation_fold"].isna().any():
    raise ValueError("Incomplete Notebook 02 person/fold assignments.")

dev = index.loc[index.role == "Development"].copy()
assert not set(dev.subject_id) & set(roles.loc[roles.role == "Test", "subject_id"])
print("Development people:", dev.subject_id.nunique(), "| windows:", len(dev))
print("Independent Development folds:", sorted(dev.validation_fold.unique()))


Development people: 35 | windows: 35000
Independent Development folds: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


**03. Recover Aligned Statistical and Temporal Inputs**

Statistical features use only the original two sensor modalities and are already aligned by the immutable `window_row`. Neural sequences are generated lazily from the same original source-sample intervals. Missing local numerical caches are reconstructed from the official source rather than selecting a different cohort or historical output.

In [4]:
# ============ 03. Recover Aligned Statistical and Temporal Inputs ============

emg_features, acc_features = ensure_numerical_cache(root, index, allow_source_download=True)
print("Feature array shapes:", emg_features.shape, acc_features.shape)
print("Fixed task classes:", classes.tolist())


Feature array shapes: (40000, 48) (40000, 96)
Fixed task classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]


**04. Define Matching Classical Model Families**

The majority-prior classifier is a source-independent sanity benchmark. Each sensing modality receives the same fixed Logistic Regression and Random Forest configurations. Within each fold, learned scalers and class weights are fitted on Training participants only; probabilities retain one fixed original-class column order.

In [5]:
# =============== 04. Define Matching Classical Model Families ================

CLASSICAL_IDS = ("B0_prior", "B1_logreg", "B1_rf", "B2_logreg", "B2_rf")
SEED = 42
common_folds = sorted(dev.validation_fold.dropna().unique().astype(int).tolist())
probabilities = {name: np.full((len(index), len(classes)), np.nan, np.float32)
                 for name in (*CLASSICAL_IDS, *UNIMODAL_MODELS)}
classical_metrics, classical_predictions, fit_record = [], [], []
for fold in common_folds:
    fit_rows = dev.loc[dev.validation_fold != fold].copy()
    val_rows = dev.loc[dev.validation_fold == fold].copy()
    if set(fit_rows.subject_id) & set(val_rows.subject_id):
        raise ValueError("A participant appears in both Training and outer Validation.")
    for name in CLASSICAL_IDS:
        started = time.perf_counter()
        model = build_classical(name, seed=SEED)
        model.fit(classical_features(name, emg_features[fit_rows.window_row],
                                     acc_features[fit_rows.window_row]),
                  fit_rows.label.to_numpy(int))
        proba = aligned_probabilities(
            model, classical_features(name, emg_features[val_rows.window_row],
                                      acc_features[val_rows.window_row]), classes,
        )
        seconds = time.perf_counter() - started
        probabilities[name][val_rows.window_row.to_numpy(int)] = proba
        metric = participant_metrics(val_rows.reset_index(drop=True), proba, classes,
                                     model_id=name, fold=fold, seed=SEED)
        metric["fit_and_prediction_seconds"] = seconds
        classical_metrics.append(metric)
        classical_predictions.append(pd.DataFrame({
            "window_key": val_rows.window_key.to_numpy(),
            "subject_id": val_rows.subject_id.to_numpy(),
            "validation_fold": fold, "model_id": name,
            "true_label": val_rows.label.to_numpy(int),
            "predicted_label": predictions_from_probabilities(proba, classes),
        }))
    print(f"Classical grouped fold {fold} complete")
classical_metrics = pd.concat(classical_metrics, ignore_index=True)
pd.concat(classical_predictions, ignore_index=True).to_csv(
    output_dir / "03_oof_predictions.csv", index=False)
classical_metrics.to_csv(output_dir / "03_dev_unimodal_metrics.csv", index=False)
display(classical_metrics.groupby("model_id", as_index=False).macro_f1.mean()
        .sort_values("macro_f1", ascending=False))


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 0 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 1 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 2 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 3 complete


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Classical grouped fold 4 complete


,model_id,macro_f1
2,B1_rf,0.234836
1,B1_logreg,0.208411
4,B2_rf,0.128978
3,B2_logreg,0.103237
0,B0_prior,0.034971


**05. Train Unimodal CNN, TCN and BiGRU Architectures**

All six neural arms consume the same original source intervals and person folds. Each outer Training fold internally reserves Training-only participants for early stopping, fits channel normalisation on the remaining Training windows and uses class-aware loss. The outer Validation fold is used for model comparison only. The programme records actual fit time, model capacity, best epoch and random seed.

In [ ]:
# ============ 05. Train Unimodal CNN, TCN and BiGRU Architectures ============

NEURAL_EPOCHS = 10
NEURAL_PATIENCE = 3
NEURAL_BATCH_SIZE = 96
neural_metrics, neural_predictions, neural_histories = [], [], []
for fold in common_folds:
    fit_rows = dev.loc[dev.validation_fold != fold].copy()
    val_rows = dev.loc[dev.validation_fold == fold].copy()
    for name in UNIMODAL_MODELS:
        model, stats, history, info = fit_neural(
            name, fit_rows, processed_dir, classes, device=device,
            seed=SEED, epochs=NEURAL_EPOCHS,
            patience=NEURAL_PATIENCE, batch_size=NEURAL_BATCH_SIZE,
        )
        proba = predict_neural(name, model, val_rows, processed_dir,
                               classes, stats, device=device)
        probabilities[name][val_rows.window_row.to_numpy(int)] = proba
        metric = participant_metrics(val_rows.reset_index(drop=True), proba,
                                     classes, model_id=name, fold=fold, seed=SEED)
        for key, value in info.items():
            if key not in ("seed", "model_id"):
                metric[key] = value
        neural_metrics.append(metric)
        history["validation_fold"] = fold
        neural_histories.append(history)
        neural_predictions.append(pd.DataFrame({
            "window_key": val_rows.window_key.to_numpy(),
            "subject_id": val_rows.subject_id.to_numpy(),
            "validation_fold": fold, "model_id": name,
            "true_label": val_rows.label.to_numpy(int),
            "predicted_label": predictions_from_probabilities(proba, classes),
        }))
        del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"Neural grouped fold {fold} complete")
neural_metrics = pd.concat(neural_metrics, ignore_index=True)
neural_history = pd.concat(neural_histories, ignore_index=True)
neural_metrics.to_csv(output_dir / "03_dev_neural_metrics.csv", index=False)
neural_history.to_csv(output_dir / "03_neural_learning_history.csv", index=False)
pd.concat(neural_predictions, ignore_index=True).to_csv(
    output_dir / "03_neural_oof_predictions.csv", index=False)


**06. Preserve Matched Out-of-Fold Class Probabilities**

Each probability matrix is indexed by the original `window_row` and uses one source-derived class order. Rows belonging to Test people remain unfilled; these arrays enable fixed-weight late fusion in Notebook 04 without retraining or mismatching predictions from separate folds.

In [ ]:
# =========== 06. Preserve Matched Out-of-Fold Class Probabilities ============

for name, proba in probabilities.items():
    subset = proba[dev.window_row.to_numpy(int)]
    if not np.isfinite(subset).all() or not np.allclose(subset.sum(axis=1), 1.0, atol=1e-4):
        raise ValueError(f"Incomplete Development out-of-fold probabilities: {name}")
np.savez_compressed(output_dir / "03_oof_probabilities.npz", **probabilities)
all_metrics = pd.concat([classical_metrics, neural_metrics], ignore_index=True)
all_metrics.to_csv(output_dir / "03_all_unimodal_metrics.csv", index=False)
chosen, summary = select_model(all_metrics, list(probabilities))
summary.to_csv(output_dir / "03_unimodal_model_summary.csv", index=False)
print("Development-selected strongest unimodal model:", chosen)
display(summary)


**07. Assess Selected Neural Models Across Three Seeds**

The principal matched benchmark uses a single prespecified seed across all folds. Additional seed runs examine stochastic training variability for the best Development neural model of each modality; they do not change the primary model-selection criterion or increase the number of independent participants.

In [ ]:
# =========== 07. Assess Selected Neural Models Across Three Seeds ============

RUN_FINALIST_ROBUSTNESS = True
EXTRA_SEEDS = (43, 44)
finalist_ids = [select_model(neural_metrics,
    [name for name in UNIMODAL_MODELS if f"_{modality}_" in name])[0]
    for modality in ("emg", "acc")]
if RUN_FINALIST_ROBUSTNESS:
    extra_metrics, extra_curves = [], []
    for name in finalist_ids:
        for replicate_seed in EXTRA_SEEDS:
            for fold in common_folds:
                fitting = dev.loc[dev.validation_fold != fold]
                validation = dev.loc[dev.validation_fold == fold]
                model, stats, history, info = fit_neural(
                    name, fitting, processed_dir, classes, device=device,
                    seed=replicate_seed, epochs=NEURAL_EPOCHS,
                    patience=NEURAL_PATIENCE, batch_size=NEURAL_BATCH_SIZE,
                )
                probs = predict_neural(name, model, validation, processed_dir,
                                       classes, stats, device=device)
                scores = participant_metrics(validation.reset_index(drop=True),
                    probs, classes, model_id=name, fold=fold, seed=replicate_seed)
                scores["fit_seconds"] = info["fit_seconds"]
                extra_metrics.append(scores)
                history["validation_fold"] = fold
                extra_curves.append(history)
                del model
    pd.concat(extra_metrics, ignore_index=True).to_csv(
        output_dir / "03_neural_multiseed_metrics.csv", index=False)
    pd.concat(extra_curves, ignore_index=True).to_csv(
        output_dir / "03_neural_multiseed_history.csv", index=False)
    print("Completed additional seeds for neural finalists:", finalist_ids)


**08. Analyse Unimodal Class Errors and Learning Curves**

Show actual Development participant-macro F1 and inner Training stopping behaviour. Class-specific errors are derived from the Development out-of-fold predictions of the strongest unimodal model; they do not describe held-out Test results.

In [ ]:
# =========== 08. Analyse Unimodal Class Errors and Learning Curves ===========

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
ordering = summary.model_id.tolist()
axes[0].barh(ordering[::-1], summary.mean_subject_macro_f1.to_numpy()[::-1])
axes[0].set(xlabel="Development participant-mean macro F1", title="Matched unimodal comparison")
curves = neural_history.groupby("epoch", as_index=False).agg(
    train_loss=("train_loss", "mean"))
axes[1].plot(curves.epoch, curves.train_loss, marker="o")
axes[1].set(xlabel="Training epoch", ylabel="Mean training loss", title="Neural optimisation (all completed runs)")
for ax in axes: ax.grid(alpha=0.2)
save_figure(fig, output_dir / "03_learning_curves.png")
plt.show()
selected_rows = dev.sort_values("window_row")
selected_proba = probabilities[chosen][selected_rows.window_row.to_numpy(int)]
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(selected_rows.label.to_numpy(int),
    predictions_from_probabilities(selected_proba, classes), labels=classes)
fig, ax = plt.subplots(figsize=(8, 6), layout="constrained")
im = ax.imshow(cm, aspect="auto")
ax.set(xlabel="Predicted source label", ylabel="True source label",
       title=f"Development out-of-fold errors: {chosen}")
ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90)
ax.set_yticks(np.arange(len(classes)), labels=classes)
fig.colorbar(im, ax=ax, label="Window count")
save_figure(fig, output_dir / "03_class_errors.png")
plt.show()


**09. Current Outputs and Scientific Interpretation**

Development evidence identifies the unimodal classical and neural candidates for paired fusion comparison in Notebook 04. The full probability arrays, model recipes, folds and training metadata are retained without using reserved Test participants.

In [ ]:
# ============= 09. Current Outputs and Scientific Interpretation =============

display(pd.DataFrame({"output_file": [
    p.name for p in sorted(output_dir.glob("%02d_*" % NOTEBOOK_NUMBER))
    if p.is_file()
]}))
download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
